# HMTV / FTY Spider JAR 逆向分析环境（Google Colab）

用途：对 `https://ches2010.github.io/tvbox/0821.json` 引用的 `spider` 文件做**静态分析**，并尝试判断加密容器能否在纯 Linux 环境下还原。

## 能力边界（先读）
- Colab 是 x86_64 Linux，**不能直接运行 ARM / AArch64 的 Android `.so`**。
- Colab 未提供 Android 模拟器、Frida/Xposed 动态环境；本 notebook 只做静态分析与可选的 Unicorn 指令模拟骨架。
- 分析第三方加固/加密代码可能涉及法律与合规问题，请确保你有权分析该文件。
- 不能保证破解；本报告已给出的证据表明真实爬虫逻辑在加密 payload 中。

开源仓库：https://github.com/ches2010/HMTV


In [ ]:
# 1. 安装分析工具（Colab 免费版即可）
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless unzip git binutils file >/dev/null 2>&1
!pip install -q requests tqdm pyelftools capstone unicorn

import os, sys, json, zipfile, hashlib, struct, re, subprocess, textwrap, requests
os.makedirs('/content/fty', exist_ok=True)
print('python', sys.version.split()[0])
!java -version 2>&1 | head -n 2


In [ ]:
# 2. 安装 JADX（从 GitHub 最新 release 动态取包）
rel = requests.get('https://api.github.com/repos/skylot/jadx/releases/latest', timeout=30).json()
asset = next(a for a in rel['assets'] if a['name'].startswith('jadx-') and a['name'].endswith('.zip'))
print('jadx', rel['tag_name'], asset['browser_download_url'])
r = requests.get(asset['browser_download_url'], timeout=120)
open('/tmp/jadx.zip','wb').write(r.content)
!rm -rf /opt/jadx && mkdir -p /opt/jadx && unzip -oq /tmp/jadx.zip -d /opt/jadx
!chmod +x /opt/jadx/bin/jadx
!/opt/jadx/bin/jadx --version


In [ ]:
# 3. 下载并校验目标文件
CONFIG_URL = 'https://ches2010.github.io/tvbox/0821.json'
JAR_URL    = 'https://ches2010.github.io/tvbox/jar/fan.txt'
cfg = requests.get(CONFIG_URL, timeout=60).json()
print('spider field:', cfg.get('spider'))
raw = requests.get(JAR_URL, timeout=120).content
open('/content/fty/fan.jar','wb').write(raw)
print('size', len(raw), 'md5', hashlib.md5(raw).hexdigest())
print('declared md5 in config:', cfg['spider'].split(';md5;')[-1])
print('zip magic', raw[:4])


In [ ]:
# 4. 展开 JAR，检查结构
z = zipfile.ZipFile('/content/fty/fan.jar')
print('\n'.join(f'{i.filename:40s} {i.file_size:>9d}' for i in z.infolist()))
for name in z.namelist():
    out = '/content/fty/' + os.path.basename(name)
    open(out,'wb').write(z.read(name))
    print('saved', out)


In [ ]:
# 5. 自写轻量 DEX 解析器：列出类、方法、字符串（不依赖 Java）
def uleb(b, off):
    res = 0; shift = 0
    while True:
        x = b[off]; off += 1
        res |= (x & 0x7f) << shift
        if not (x & 0x80): break
        shift += 7
    return res, off

def parse_dex(dex):
    u32 = lambda o: struct.unpack_from('<I', dex, o)[0]
    u16 = lambda o: struct.unpack_from('<H', dex, o)[0]
    ss, so = u32(0x38), u32(0x3c); ts, to = u32(0x40), u32(0x44)
    ps, po = u32(0x48), u32(0x4c); ms, mo = u32(0x58), u32(0x5c)
    strings = []
    for i in range(ss):
        p = u32(so+4*i); n, q = uleb(dex, p)
        strings.append(dex[q:q+n].decode('utf-8','replace'))
    types = [strings[u32(to+4*i)] for i in range(ts)]
    def params(off):
        if off == 0: return []
        c = u32(off); return [types[u16(off+4+2*i)] for i in range(c)]
    protos = []
    for i in range(ps):
        p = po+12*i; protos.append((types[u32(p+4)], params(u32(p+8))))
    methods = []
    for i in range(ms):
        p = mo+8*i; ci, pi = u16(p), u16(p+2); ni = u32(p+4)
        ret, pars = protos[pi]
        methods.append((types[ci], strings[ni], ret, pars))
    return {'strings':strings, 'types':types, 'methods':methods}

dex = parse_dex(open('/content/fty/classes.dex','rb').read())
print('strings', len(dex['strings']), 'types', len(dex['types']), 'methods', len(dex['methods']))
guards = sorted(set(m[0] for m in dex['methods'] if 'catvod/spider' in m[0] and m[0].endswith('Guard;')))
print('Guard classes:', len(guards))
print('\n'.join(guards[:60]))
print('\nDexNative methods:')
for m in dex['methods']:
    if 'DexNative' in m[0]: print(' ', m[1], m[2], m[3])


In [ ]:
# 6. 用 JADX 反编译外层 DEX
!rm -rf /content/out/jadx && mkdir -p /content/out/jadx
!/opt/jadx/bin/jadx -d /content/out/jadx --no-res /content/fty/classes.dex 2>&1 | tail -n 5
print('decompiled files:', sum(len(f) for _,_,f in os.walk('/content/out/jadx')))
for p in ['DexNative.java','Init.java','BaseSpiderGuard.java','DouDouGuard.java']:
    hits = [os.path.join(dp,p) for dp,_,fs in os.walk('/content/out/jadx') if p in fs]
    print(p, hits)


In [ ]:
# 7. 原生库静态分析（ELF 头 / 架构 / 字符串 / 导出函数）
from elftools.elf.elffile import ELFFile
from capstone import Cs, CS_ARCH_ARM, CS_ARCH_ARM64, CS_MODE_ARM, CS_MODE_LITTLE_ENDIAN

for fn in ['ftyguard_v7.so','ftyguard_v8.so']:
    path='/content/fty/'+fn
    b=open(path,'rb').read()
    print('===',fn,len(b),b[:8])
    elf=ELFFile(open(path,'rb'))
    print('machine', elf.header['e_machine'], 'class', elf.elfclass)
    sec=elf.get_section_by_name('.dynsym')
    syms=[s.name for s in sec.iter_symbols()] if sec else []
    print('dynsym sample', [s for s in syms if s][:40])
    print('JNI_OnLoad' in syms, [s for s in syms if 'datadiv' in s or 'decode' in s][:20])
    strs=[s.decode('ascii','replace') for s in re.findall(rb'[ -~]{6,}', b)]
    print('obfuscator?', any('Obfuscator' in s for s in strs))


In [ ]:
# 8. 加密容器 ftyshinidie.guard 结构探测
b=open('/content/fty/ftyshinidie.guard','rb').read()
print('size',len(b),'head',b[:32])
print('local header', struct.unpack_from('<4s5H3L2H', b, 0))
print('central-dir sig swapped count', b.count(b'PK\x02\x01'), 'eocd swapped count', b.count(b'PK\x06\x05'))
print('normal ZIP?', zipfile.is_zipfile('/content/fty/ftyshinidie.guard'))
#  locust: 常见 ZipCrypto 口令尝试（预期失败，用来证明不是标准加密 zip）
def zipcrypto_decrypt(data, pwd):
    keys=[0x12345678,0x23456789,0x34567890]
    def cr32(c,k): return ((c>>1) ^ (0xEDB88320 if c&1 else 0) ^ k) & 0xffffffff
    def upd(k,c): return cr32((k + c)&0xffffffff, k)
    for ch in pwd.encode():
        keys[0]=upd(keys[0],ch)
    keys[1]=upd(keys[1],keys[2]&0xff)
    out=bytearray()
    for c in data:
        k=((keys[2]&0xffff)|2)*(((keys[2]&0xffff)|2)^1)>>8
        c^= k&0xff
        keys[0]=upd(keys[0],c); keys[1]=upd(keys[1],keys[0]&0xff); keys[2]=upd(keys[2],keys[1]&0xff)
        out.append(c)
    return bytes(out)
name_len, extra_len = struct.unpack_from('<HH', b, 26)
data_off = 30+name_len+extra_len
comp_size = struct.unpack_from('<I', b, 18)[0]
enc = b[data_off:data_off+comp_size]
for pwd in [b'', b'fty', b'ftyguard', b'ftyshinidie', b'123456', b'fan', b'tvbox']:
    d=zipcrypto_decrypt(enc[:64], pwd)
    ok = d[:2] in (b'\x78\x9c', b'\x78\xda', b'\x78\x01')
    print(pwd, d[:8].hex(), 'deflate-looking', ok)


In [ ]:
# 9. 结论输出（供存档）
report = {
 'config_md5_declared': cfg['spider'].split(';md5;')[-1],
 'jar_md5_actual': hashlib.md5(raw).hexdigest(),
 'outer_dex_classes': len(dex['types']),
 'outer_dex_methods': len(dex['methods']),
 'guard_classes': len(guards),
 'encrypted_payload_size': len(b),
 'is_standard_zip': zipfile.is_zipfile('/content/fty/ftyshinidie.guard'),
 'note': 'Real spiders are inside an encrypted payload; key/algorithm lives in ARM .so; Colab cannot run it directly.'
}
print(json.dumps(report, indent=2, ensure_ascii=False))
with open('/content/fty/colab_report.json','w') as f: json.dump(report,f,indent=2,ensure_ascii=False)
